In [1]:
# Estrategia Lay 0x1

import pandas as pd
import numpy as np

In [71]:
data = pd.read_csv("../data_total/dados_betfair.csv", sep=";")

In [72]:
# Filtar colunas para análise
datatest = data[['League', 'Home', 'Away', 'Goals_H_HT', 'Goals_A_HT', 'Goals_H_FT', 'Goals_A_FT', 'Min_Goals_H', 'Odd_H_Back', 'Odd_A_Back', 'Odd_CS_0x1_Lay']].copy()

#datatest.to_csv("TEBF002_Lay_0x1.csv", sep=";", index=False)

# Verificar se o placar FT foi 0x1
datatest['WCS'] = datatest.apply(lambda row: 0 if row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 1 else 1, axis=1)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_CS_0x1_Lay'] - 1) if row['WCS'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

# Adicionar coluna com o minuto do primeiro gol
datatest['Min_Goal_0x0'] = np.where(
    (datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 0),
    datatest['Min_Goals_H'].apply(lambda x: x[1:3] if len(x) > 0 else 0),
    0
)

# Alterar [ para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].str.replace(']', '0')

# Alterar NaN para 0
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].fillna(0)

# Transformar a coluna em inteiro
datatest['Min_Goal_0x0'] = datatest['Min_Goal_0x0'].astype(int)


datatest.head(15)

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Min_Goals_H,Odd_H_Back,Odd_A_Back,Odd_CS_0x1_Lay,WCS,Profit,Min_Goal_0x0
0,SPAIN 1,Mallorca,Granada CF,0,0,1,0,[85],1.88,5.30,14.0,1,0.94,85
1,SPAIN 1,Osasuna,Real Madrid,1,2,2,4,"[7, 90]",6.60,1.58,8.6,1,0.94,0
2,SPAIN 1,Getafe,Girona,1,0,1,0,[33],3.35,2.34,11.0,1,0.94,0
3,SPAIN 1,Ath Bilbao,Alaves,2,0,2,0,"[32, 37]",1.59,7.60,20.0,1,0.94,0
4,ENGLAND 1,Fulham,Tottenham,1,0,3,0,"[42, 49, 61]",3.50,2.10,15.5,1,0.94,0
5,ENGLAND 1,Burnley,Brentford,1,0,2,1,"[10, 62]",3.35,2.26,12.0,1,0.94,0
6,ENGLAND 1,Luton,Nottingham,0,1,1,1,[89],2.88,2.56,14.0,1,0.94,0
7,ITALY 1,Udinese,Torino,0,1,0,2,[],2.98,2.92,8.6,1,0.94,0
8,ITALY 1,Monza,Cagliari,1,0,1,0,[41],2.12,3.85,14.5,1,0.94,0
9,ITALY 1,Salernitana,Lecce,0,1,0,1,[],3.10,2.60,9.4,0,-8.40,0


In [73]:
# Função para criar faixas de odds
def criar_faixa_h_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
    
def criar_faixa_a_back(odd):
    if odd < 1.80:
        return '1.50-1.79'
    elif odd < 2.10:
        return '1.80-2.09'
    elif odd < 2.50:
        return '2.10-2.49'
    elif odd < 3.00:
        return '2.50-2.99'
    elif odd < 3.50:
        return '3.00-3.49'
    elif odd < 4.00:
        return '3.50-3.99'
    elif odd < 5.00:
        return '4.00-4.99'
    else:
        return '5.00+'
        
def criar_faixa_cs_lay(odd):
    if odd < 9.0:
        return '8.0-8.9'
    elif odd < 10.0:
        return '9.0-9.9'
    elif odd < 11.0:
        return '10.0-10.9'
    elif odd < 12.0:
        return '11.0-11.9'
    elif odd < 13.0:
        return '12.0-12.9'
    elif odd < 14.0:
        return '13.0-13.9'
    elif odd < 15.0:
        return '14.0-14.9'
    elif odd < 16.0:
        return '15.0-15.9'
    elif odd < 18.0:
        return '16.0-17.9'
    elif odd < 20.0:
        return '18.0-19.9'
    else:
        return '20.0+'
    
# Aplicar as funções às colunas correspondentes
datatest['Faixa_Odd_H_Back'] = datatest['Odd_H_Back'].apply(criar_faixa_h_back)
datatest['Faixa_Odd_A_Back'] = datatest['Odd_A_Back'].apply(criar_faixa_a_back)
datatest['Faixa_Odd_CS_0x1_Lay'] = datatest['Odd_CS_0x1_Lay'].apply(criar_faixa_cs_lay)

# Agrupars por faixas e calcular estatísticas
print("\n📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):")
print("-" * 80)

agrupando_faixas = datatest.groupby(['Faixa_Odd_H_Back', 'Faixa_Odd_A_Back', 'Faixa_Odd_CS_0x1_Lay']).agg(
    Total_Jogos=('WCS', 'count'),
    Jogos_0x1=('WCS', 'sum'),
    Percentual_Acerto=('WCS', lambda x: (x.sum() / len(x) * 100)),
    Lucro_Total=('Profit', 'sum')
).reset_index()

# Arredondar valores
agrupando_faixas['Percentual_Acerto'] = agrupando_faixas['Percentual_Acerto'].round(2)

# Agrupar por lucro Total
agrupando_faixas = agrupando_faixas.sort_values(by='Lucro_Total', ascending=False)

agrupando_faixas.head(10)



📊 ANÁLISE POR FAIXAS (recomendado para amostras maiores):
--------------------------------------------------------------------------------


,Faixa_Odd_H_Back,Faixa_Odd_A_Back,Faixa_Odd_CS_0x1_Lay,Total_Jogos,Jogos_0x1,Percentual_Acerto,Lucro_Total
25,1.80-2.09,4.00-4.99,20.0+,74,73,98.65,48.62
20,1.80-2.09,4.00-4.99,13.0-13.9,53,52,98.11,36.88
54,2.10-2.49,3.50-3.99,12.0-12.9,73,70,95.89,32.30
24,1.80-2.09,4.00-4.99,18.0-19.9,53,52,98.11,30.88
57,2.10-2.49,3.50-3.99,15.0-15.9,30,30,100.00,28.20
80,2.50-2.99,2.50-2.99,11.0-11.9,40,39,97.50,26.16
31,1.80-2.09,5.00+,15.0-15.9,41,40,97.56,23.60
30,1.80-2.09,5.00+,14.0-14.9,40,39,97.50,23.16
64,2.10-2.49,4.00-4.99,11.0-11.9,54,51,94.44,17.94
58,2.10-2.49,3.50-3.99,16.0-17.9,37,36,97.30,17.84


In [74]:
# Selecionar apenas as 8 linhas com maior lucro total
top_10_lucro = agrupando_faixas.head(10)

# Total de jogos
total_jogos = top_10_lucro['Total_Jogos'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = top_10_lucro['Jogos_0x1'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🎯 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = top_10_lucro['Lucro_Total'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")


📈 Total de Jogos: 495
✅ Total de Acertos: 482
🎯 Percentual de Acerto: 97.37%
💰 Total de Lucro: 285.58


In [76]:
condicoes = [
    # Linha 1
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'] >= 20.0)),
    
    # Linha 2
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(13.0, 13.9))),
    
    # Linha 3
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(12.0, 12.9))),
    
    # Linha 4
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(18.0, 19.9))),
    
    # Linha 5
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 6
    ((datatest['Odd_H_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_A_Back'].between(2.50, 2.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 7
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(15.0, 15.9))),
    
    # Linha 8
    ((datatest['Odd_H_Back'].between(1.80, 2.09)) & 
     (datatest['Odd_A_Back'] >= 5.00) & 
     (datatest['Odd_CS_0x1_Lay'].between(14.0, 14.9))),
    
    # Linha 9
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(4.00, 4.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(11.0, 11.9))),
    
    # Linha 10
    ((datatest['Odd_H_Back'].between(2.10, 2.49)) & 
     (datatest['Odd_A_Back'].between(3.50, 3.99)) & 
     (datatest['Odd_CS_0x1_Lay'].between(16.0, 17.9)))
]

# Aplicar as condições: Bet = 1 se qualquer uma das condições for verdadeira
datatest['Bet'] = np.where(pd.concat(condicoes, axis=1).any(axis=1), 1, 0)

# Calcular o lucro para cada jogo com base nas condições
datatest['PL'] = np.where(
    (datatest['Bet'] == 1) & (datatest['WCS'] == 1),
    datatest['Profit'],
    np.where(
        (datatest['Bet'] == 1) & (datatest['WCS'] == 0),
        - datatest['Odd_CS_0x1_Lay'] + 1,
        0
    )
)

datatest['PL_ACC'] = datatest['PL'].cumsum()

# Jogos No HT diferente de 0x0
datatest['HT_Dif_0x0'] = np.where((datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 0), 1, 0)

# Jogos No HT diferente de 0x1
datatest['HT_Dif_0x1'] = np.where((datatest['Goals_H_HT'] == 0) & (datatest['Goals_A_HT'] == 1), 1, 0)

# Jogos no HT sem ser 0x0 ou 0x1
datatest['HT_Dif_0x0_0x1'] = np.where((datatest['HT_Dif_0x0'] == 1) | (datatest['HT_Dif_0x1'] == 1), 0, 1)

# Jogos no FT 0x0
datatest['FT_Dif_0x0'] = np.where((datatest['Goals_H_FT'] == 0) & (datatest['Goals_A_FT'] == 0), 1, 0)

# Jogos no FT 0x1
datatest['FT_Dif_0x1'] = np.where((datatest['Goals_H_FT'] == 0) & (datatest['Goals_A_FT'] == 1), 1, 0)

# Jogos mo HT 0x0 e no FT 0x1
datatest['HT_0x0_FT_0x1'] = np.where((datatest['HT_Dif_0x0'] == 1) & (datatest['FT_Dif_0x1'] == 1), 1, 0)

# Jogos no HT 0x1 e diferente no FT
datatest['HT_0x1_FT_Dif_0x1'] = np.where((datatest['HT_Dif_0x1'] == 1) & (datatest['FT_Dif_0x1'] == 0), 1, 0)

    
datatest.head(15)

#datatest.to_csv("TEBF002_Lay_0x1_ANALISE.csv", sep=";", index=False)
    

,League,Home,Away,Goals_H_HT,Goals_A_HT,Goals_H_FT,Goals_A_FT,Min_Goals_H,Odd_H_Back,Odd_A_Back,...,Bet,PL,PL_ACC,HT_Dif_0x0,HT_Dif_0x1,HT_Dif_0x0_0x1,FT_Dif_0x0,FT_Dif_0x1,HT_0x0_FT_0x1,HT_0x1_FT_Dif_0x1
0,SPAIN 1,Mallorca,Granada CF,0,0,1,0,[85],1.88,5.30,...,1,0.94,0.94,1,0,0,0,0,0,0
1,SPAIN 1,Osasuna,Real Madrid,1,2,2,4,"[7, 90]",6.60,1.58,...,0,0.00,0.94,0,0,1,0,0,0,0
2,SPAIN 1,Getafe,Girona,1,0,1,0,[33],3.35,2.34,...,0,0.00,0.94,0,0,1,0,0,0,0
3,SPAIN 1,Ath Bilbao,Alaves,2,0,2,0,"[32, 37]",1.59,7.60,...,0,0.00,0.94,0,0,1,0,0,0,0
4,ENGLAND 1,Fulham,Tottenham,1,0,3,0,"[42, 49, 61]",3.50,2.10,...,0,0.00,0.94,0,0,1,0,0,0,0
5,ENGLAND 1,Burnley,Brentford,1,0,2,1,"[10, 62]",3.35,2.26,...,0,0.00,0.94,0,0,1,0,0,0,0
6,ENGLAND 1,Luton,Nottingham,0,1,1,1,[89],2.88,2.56,...,0,0.00,0.94,0,1,0,0,0,0,1
7,ITALY 1,Udinese,Torino,0,1,0,2,[],2.98,2.92,...,0,0.00,0.94,0,1,0,0,0,0,1
8,ITALY 1,Monza,Cagliari,1,0,1,0,[41],2.12,3.85,...,0,0.00,0.94,0,0,1,0,0,0,0
9,ITALY 1,Salernitana,Lecce,0,1,0,1,[],3.10,2.60,...,0,0.00,0.94,0,1,0,0,1,0,0


In [80]:
# Total de jogos
total_jogos = datatest['Bet'].sum()
print(f"📈 Total de Jogos: {total_jogos}")

# Total de acertos
total_acertos = datatest[datatest['Bet'] == 1]['WCS'].sum()
print(f"✅ Total de Acertos: {total_acertos}")

# Percentual de acerto
percentual_acerto = (total_acertos / total_jogos) * 100
print(f"🔍 Percentual de Acerto: {percentual_acerto:.2f}%")

# Total de Lucro
total_lucro = datatest['PL'].sum()
print(f"💰 Total de Lucro: {total_lucro:.2f}")

# Odd Média
odd_media = datatest[datatest['Bet'] == 1]['Odd_CS_0x1_Lay'].mean()
print(f"📊 Odd Média: {odd_media:.2f}")

# Drawndown
drawdown = datatest['PL_ACC'] - datatest['PL_ACC'].cummax()
max_drawdown = drawdown.min()  # ou .max() dependendo da convenção
print(f"📉 Max Drawdown: {max_drawdown:.2f}%")

# Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1
jogos_ht_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1: {jogos_ht_0x0}")

# Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1
jogos_ht_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1: {jogos_ht_0x1}")

# Jogos no HT sem ser 0x0 ou 0x1
jogos_ht_0x0_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT sem ser 0x0 ou 0x1, se Bet == 1 & HT_Dif_0x0_0x1 == 1: {jogos_ht_0x0_0x1}")

# Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1
jogos_ft_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['FT_Dif_0x0'] == 1)].shape[0]
print(f"📊 Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1: {jogos_ft_0x0}")

# Jogos no HT 0x0 e No FT 0x1
jogos_ht_0x0_ft_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['HT_0x0_FT_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x0 e No FT 0x1, se Bet == 1 & HT_0x0_FT_0x1 == 1: {jogos_ht_0x0_ft_0x1}")

# Jogo no HT 0x1 eno FT 0x1
jogos_ht_0x1_ft_0x1 = datatest[(datatest['Bet'] == 1) & (datatest['FT_Dif_0x1'] == 1)].shape[0]
print(f"📊 Jogos no HT 0x1 e No FT 0x1, se Bet == 1 & HT_0x1_FT_Dif_0x1 == 1: {jogos_ht_0x1_ft_0x1}")

# Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0
media_minuto_gol_0x0 = datatest[(datatest['Bet'] == 1) & (datatest['HT_Dif_0x0_0x1'] == 0)]['Min_Goal_0x0'].mean()
print(f"📊 Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0: {media_minuto_gol_0x0:.2f}")


📈 Total de Jogos: 495
✅ Total de Acertos: 482
🔍 Percentual de Acerto: 97.37%
💰 Total de Lucro: 285.58
📊 Odd Média: 21.23
📉 Max Drawdown: -23.42%
📊 Jogos no HT 0x0, se Bet == 1 & HT_Dif_0x0 == 1: 151
📊 Jogos no HT 0x1, se Bet == 1 & HT_Dif_0x1 == 1: 64
📊 Jogos no HT sem ser 0x0 ou 0x1, se Bet == 1 & HT_Dif_0x0_0x1 == 1: 280
📊 Jogos no FT 0x0, se Bet == 1 & FT_Dif_0x0 == 1: 39
📊 Jogos no HT 0x0 e No FT 0x1, se Bet == 1 & HT_0x0_FT_0x1 == 1: 8
📊 Jogos no HT 0x1 e No FT 0x1, se Bet == 1 & HT_0x1_FT_Dif_0x1 == 1: 13
📊 Media do minuto gol 0x0 , se Bet == 1 e HT_Dif_0x1_0x0 == 0: 29.02


In [ ]:
# Se 0x0 HT Red Max 6% | Min 4% ***Acima de 6% Red Sair da Posição
# Se 0x0 Até no Min. 65 Red Max 7% ***Acima de 7% Red sair da Posição
# Se 0x0 Até no Min. 80 Neutro

# Se 0x1 Ht Red Max 15% | ***Acima de 15% Red Sair da Posição
# Se 0x1 Até no Min. 65 Red Max 25% | ***Acima de 25% Red Sair da Posição

